# RealFill: Personalized Text-to-Image Inpainting

**RealFill** ([arXiv:2309.16668](https://arxiv.org/abs/2309.16668)) is a method to personalize text-to-image inpainting models like Stable Diffusion inpainting, given just 1–5 images of a scene.

This notebook reproduces the training pipeline using LoRA fine-tuning on the UNet and text encoder via the `diffusers` + `peft` libraries.

> **Note:** This is an unofficial implementation built on top of the DreamBooth training script from `diffusers`. The repo is at [github.com/thuanz123/realfill](https://github.com/thuanz123/realfill).

## 1. Setup: Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/thuanz123/realfill.git

In [ ]:
%cd realfill
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -r requirements.txt
!pip install matplotlib bitsandbytes xformers  # needed for the visualization cell

## 2. Configure Accelerate

Accelerate is used for distributed/mixed-precision training. In a notebook environment (non-interactive shell), use `write_basic_config()` for a default configuration.

In [ ]:
from accelerate.utils import write_basic_config

write_basic_config()

In [ ]:
import torch
print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Training RealFill

> **Prerequisites:** A HuggingFace token (`HF_TOKEN`) with access to `stabilityai/stable-diffusion-2-inpainting` is required to download the base model. Set it as an environment variable or log in with `huggingface_hub login`. A GPU with CUDA is strongly recommended for training.

The training script fine-tunes **both the UNet and the text encoder** using LoRA (via `peft`). It uses the reference images in `data/<scene>/ref/` and the target image + mask in `data/<scene>/target/.`

In [ ]:
%cd /content/realfill/data/flowerwoman
%ls

MODEL_NAME = "sd2-community/stable-diffusion-2-inpainting"

# Verify data structure
import pathlib
ref_dir = pathlib.Path("/content/realfill/data/flowerwoman/ref")
target_dir = pathlib.Path("/content/realfill/data/flowerwoman/target")
print(f"Reference images: {list(ref_dir.iterdir())}")
print(f"Target files: {list(target_dir.iterdir())}")

In [ ]:
import os
import sys

os.chdir("/content/realfill")

TRAIN_DIR  = "data/flowerwoman"
OUTPUT_DIR = "flowerwoman-model"

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("Set HF_TOKEN in your environment before running this cell.")
os.environ["HF_TOKEN"] = HF_TOKEN

sys.argv = [
    "train_realfill.py",
    "--pretrained_model_name_or_path", "sd2-community/stable-diffusion-2-inpainting",
    "--train_data_dir", TRAIN_DIR,
    "--output_dir", OUTPUT_DIR,
    "--resolution", "512",
    "--train_batch_size", "16",
    "--gradient_accumulation_steps", "1",
    "--gradient_checkpointing",
    "--use_8bit_adam",
    "--set_grads_to_none",
    "--mixed_precision", "fp16",
    "--unet_learning_rate", "2e-4",
    "--text_encoder_learning_rate", "4e-5",
    "--lr_scheduler", "constant",
    "--lr_warmup_steps", "100",
    "--max_train_steps", "2000",
    "--lora_rank", "8",
    "--lora_dropout", "0.1",
    "--lora_alpha", "16",
    "--allow_tf32"
]

from train_realfill import parse_args, main
args = parse_args()
main(args)

## 4. Inference with the Trained Model

After training, the merged LoRA weights are saved to `OUTPUT_DIR`. Use `infer.py` to run inpainting on a held-out image with a mask.

The script:
1. Loads the base Stable Diffusion inpainting pipeline
2. Overwrites the UNet and text encoder with your trained LoRA weights
3. Applies the mask to the target image and fills in the masked region
4. Saves 16 output images (one per inference run)

In [ ]:
%cd /content/realfill

TRAIN_DIR  = "data/flowerwoman"
OUTPUT_DIR = "flowerwoman-model"
VAL_IMAGE  = f"{TRAIN_DIR}/target/target.png"
VAL_MASK   = f"{TRAIN_DIR}/target/mask.png"
OUT_DIR   = "./test-infer/"

import sys, os

sys.argv = [
    "infer.py",
    "--model_path", OUTPUT_DIR,
    "--validation_image", VAL_IMAGE,
    "--validation_mask", VAL_MASK,
    "--output_dir", OUT_DIR,
    "--seed", "40",
]

from infer import parse_args, main
main(parse_args())

## 5. Visualize Results

In [ ]:
%cd /content/realfill

from PIL import Image
import matplotlib.pyplot as plt
import glob, os

OUT_DIR = "./test-infer/"
results = sorted(glob.glob(os.path.join(OUT_DIR, "*.png")))

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
axes = axes.flatten()

for ax, path in zip(axes, results[:16]):
    ax.imshow(Image.open(path))
    ax.set_title(os.path.basename(path))
    ax.axis("off")

fig.suptitle("RealFill Inpainting Results", fontsize=14)
plt.tight_layout()
plt.show()

## Summary

This notebook walked through the complete RealFill pipeline:

1. **Setup** — cloned the repo and installed dependencies
2. **Accelerate config** — initialized distributed training settings
3. **Training** — fine-tuned LoRA on both UNet and text encoder using reference images
4. **Low-memory training** — added gradient checkpointing, 8-bit Adam, xFormers, and `set_grads_to_none`
5. **Inference** — loaded the trained LoRA into Stable Diffusion inpainting and ran it on a held-out image + mask
6. **Visualization** — displayed the inpainted results

### Data format expected by `RealFillDataset`

```
data/<scene>/
├── ref/               # 1–5 reference photos of the scene
│   ├── img1.jpg
│   ├── img2.jpg
│   └── ...
└── target/
    ├── target.png     # image with an area to fill
    └── mask.png       # binary mask (white = area to inpaint)
```

The training prompt is hardcoded as `"a photo of sks"` and can be changed in `train_realfill.py` (`self.train_prompt`). After training, the merged LoRA weights are saved in `output_dir/` and can be loaded by `infer.py` or any `diffusers` pipeline.